# Limpeza, Filtragem e Padronização Extrema

Este notebook lerá a base inteira, que agora estará num formato altamente indexado e leve (Parquet), realizará a limpeza do ruído e padronizará tudo estatisticamente para uso seguro nos seus modelos de Análise Multidimensional PCA/MDS.

Os algoritmos baseados em álgebra linear exigem dados sem ruídos e na mesma escala. Vamos remover as anomalias ("UNKNOWN") e aplicar o StandardScaler (Z-Score).

## 1. Importação das libs

In [28]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

SILVER_PATH = '../02_silver/formai_silver.parquet'
GOLD_PATH = '../03_gold/formai_gold.parquet'

print("Bibliotecas importadas com sucesso.")

Bibliotecas importadas com sucesso.


## 2. Leitura, Filtragem de Dados Ruidosos

In [29]:
# Carregar o dataset da camada Silver
df_silver = pd.read_parquet(SILVER_PATH)

In [30]:
df_silver.head()

,file_name,category,error_type,num_lines,cyclomatic_complexity,token_count,function_count,avg_parameter_count
0,gpt4o_mini-31631.c,UNKNOWN (time out),N/A,118.0,2.10,617.0,10.0,1.0
1,falcon180b-51215.c,UNKNOWN (time out),N/A,86.0,3.25,418.0,4.0,1.5
2,gpt35-8997.c,UNKNOWN (time out),N/A,54.0,4.00,233.0,2.0,0.5
3,gpt4o_mini-34206.c,VULNERABLE,buffer overflow on scanf,74.0,3.00,302.0,4.0,0.5
4,gpt4o_mini-34206.c,VULNERABLE,buffer overflow on scanf,74.0,3.00,302.0,4.0,0.5


In [31]:
df_silver.info()

<class 'pandas.DataFrame'>
RangeIndex: 883140 entries, 0 to 883139
Data columns (total 8 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   file_name              883140 non-null  str    
 1   category               883140 non-null  str    
 2   error_type             883140 non-null  str    
 3   num_lines              883140 non-null  float64
 4   cyclomatic_complexity  883140 non-null  float64
 5   token_count            883140 non-null  float64
 6   function_count         883140 non-null  float64
 7   avg_parameter_count    883140 non-null  float64
dtypes: float64(5), str(3)
memory usage: 99.7 MB


In [42]:
df_silver.isna().sum()

file_name                0
category                 0
error_type               0
num_lines                0
cyclomatic_complexity    0
token_count              0
function_count           0
avg_parameter_count      0
dtype: int64

## 2. Filtrando categorias ruidosas

In [45]:
total = len(df_silver)
df_silver['category'].value_counts()

category
VULNERABLE            765366
UNKNOWN (time out)     83805
NON-VULNERABLE         25674
PARSING ERROR           8295
Name: count, dtype: int64

In [54]:
print(f'% VULNERABLE: {len(df_silver[df_silver['category'] == 'VULNERABLE']) / total * 100}')
print(f'% UNKNOWN (time out): {len(df_silver[df_silver['category'] == 'UNKNOWN (time out)']) / total * 100}')
print(f'% NON-VULNERABLE: {len(df_silver[df_silver['category'] == 'NON-VULNERABLE']) / total * 100}')
print(f'% PARSING ERROR: {len(df_silver[df_silver['category'] == 'PARSING ERROR']) / total * 100}')

% VULNERABLE: 86.66417555540458
% UNKNOWN (time out): 9.489435423602147
% NON-VULNERABLE: 2.9071268428561723
% PARSING ERROR: 0.9392621781371017


In [38]:
valid_categories = ['VULNERABLE', 'NON-VULNERABLE']

Manteremos apenas falhas confirmadas e códigos seguros

In [55]:
df_gold = df_silver[df_silver['category'].isin(valid_categories)].copy()

df_gold.category.value_counts()

category
VULNERABLE        765366
NON-VULNERABLE     25674
Name: count, dtype: int64

Definindo as 5 variáveis numéricas da atividade

In [ ]:
numeric_features = [
    'num_lines', 
    'cyclomatic_complexity', 
    'token_count', 
    'function_count', 
    'avg_parameter_count'
]

### Tratando possíveis dados faltantes

In [56]:
df_gold[numeric_features].isna().sum()

num_lines                0
cyclomatic_complexity    0
token_count              0
function_count           0
avg_parameter_count      0
dtype: int64

In [57]:
df_gold[numeric_features] = df_gold[numeric_features].fillna(0)

## 3. Padronização Z-Score Extreme (Obrigatório para MDS/PCA)

In [58]:
scaler = StandardScaler()

In [59]:
df_gold[numeric_features] = scaler.fit_transform(df_gold[numeric_features])

In [61]:
display(df_gold.head())

,file_name,category,error_type,num_lines,cyclomatic_complexity,token_count,function_count,avg_parameter_count
3,gpt4o_mini-34206.c,VULNERABLE,buffer overflow on scanf,-0.495477,-0.381917,-0.665436,0.095796,-0.606073
4,gpt4o_mini-34206.c,VULNERABLE,buffer overflow on scanf,-0.495477,-0.381917,-0.665436,0.095796,-0.606073
5,gpt4o_mini-34206.c,VULNERABLE,buffer overflow on scanf,-0.495477,-0.381917,-0.665436,0.095796,-0.606073
6,gemini_pro-14924.c,VULNERABLE,dereference failure: invalid pointer,0.170739,-0.614471,0.319602,1.385231,0.522932
7,gemini_pro-14924.c,VULNERABLE,dereference failure: NULL pointer,0.170739,-0.614471,0.319602,1.385231,0.522932


## 4. Salvando o dataset final Gold em formato Parquet

In [62]:
df_gold.to_parquet(GOLD_PATH, engine='pyarrow', index=False)

print(f"Camada Gold concluída! Shape final do dataset: {df_gold.shape}")
df_gold.head()

Camada Gold concluída! Shape final do dataset: (791040, 8)


,file_name,category,error_type,num_lines,cyclomatic_complexity,token_count,function_count,avg_parameter_count
3,gpt4o_mini-34206.c,VULNERABLE,buffer overflow on scanf,-0.495477,-0.381917,-0.665436,0.095796,-0.606073
4,gpt4o_mini-34206.c,VULNERABLE,buffer overflow on scanf,-0.495477,-0.381917,-0.665436,0.095796,-0.606073
5,gpt4o_mini-34206.c,VULNERABLE,buffer overflow on scanf,-0.495477,-0.381917,-0.665436,0.095796,-0.606073
6,gemini_pro-14924.c,VULNERABLE,dereference failure: invalid pointer,0.170739,-0.614471,0.319602,1.385231,0.522932
7,gemini_pro-14924.c,VULNERABLE,dereference failure: NULL pointer,0.170739,-0.614471,0.319602,1.385231,0.522932
